In [72]:
from sklearn.datasets import fetch_california_housing

import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np
import pandas as pd 

from sklearn.model_selection import train_test_split 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline 

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor 
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score 

In [20]:
housing = fetch_california_housing(as_frame=True)
df_original = housing.frame
df = df_original.copy()

## EDA

### MedInc
MedInc has 690 entries above 8, 2 stdevs above mean, but it's progressive so doesn't seem crazy

### HouseAge
HouseAge looks fine

### AveRooms
AveRooms has 2 crazy outliers of 141 and 132, even the 60 rooms are pretty afar off from the mean+std, but it's progressive

### AveBedrms
AveBedrms has the same 2 outliers from AveRoom, index1914, index1979

### Population
Population has 2 outliers signficantly out, Index9880, Index15360

### AveOccup
AveOccup has 4 outliers, Index3364, 13034, 16669, 19006
AveOccup is highly right-skewed, with a median of 2.8 and mean of 3, std of 10, but max of 1243
the 4 index (all >100) seems highly unplausible, therefore treated as outliers and removed 

In [21]:
outliers_index = [
    1914,
    1979,
    9880,
    15360,
    3364,
    13034,
    16669,
    19006
]

print(df.loc[outliers_index])

print(df['AveOccup'].describe())
print(df['AveOccup'].sort_values(ascending=False).head(20))

        MedInc  HouseAge    AveRooms  AveBedrms  Population     AveOccup  \
1914    1.8750      33.0  141.909091  25.636364        30.0     2.727273   
1979    4.6250      34.0  132.533333  34.066667        36.0     2.400000   
9880    2.3087      11.0    5.364518   1.059684     28566.0     4.696810   
15360   2.5729      14.0    5.270497   1.010484     35682.0     7.482072   
3364    5.5179      36.0    5.142857   1.142857      4198.0   599.714286   
13034   6.1359      52.0    8.275862   1.517241      6675.0   230.172414   
16669   4.2639      46.0    9.076923   1.307692      6532.0   502.461538   
19006  10.2264      45.0    3.166667   0.833333      7460.0  1243.333333   

       Latitude  Longitude  MedHouseVal  
1914      38.91    -120.10      5.00001  
1979      38.80    -120.08      1.62500  
9880      36.64    -121.79      1.18800  
15360     33.35    -117.42      1.34400  
3364      40.41    -120.51      0.67500  
13034     38.69    -121.15      2.25000  
16669     35.32    -1

In [22]:
df = df[df['AveOccup'] <= 100]

In [39]:
y = df['MedHouseVal']
X = df.drop(['MedHouseVal'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size = 0.2,
                                                    random_state=42,
                                                   )

preprocessor = Pipeline([
    ('preprocess', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

In [63]:
#linearRegression baseline model

model_LR = Pipeline([
    ("preprecessor", preprocessor),
    ("linear_reg", LinearRegression())
])

model_LR.fit(X_train, y_train)
y_pred = model_LR.predict(X_test)

model_LR_MAE = mean_absolute_error(y_test, y_pred)
model_LR_MSE = mean_squared_error(y_test, y_pred)
model_LR_RMSE = np.sqrt(model_LR_MSE)
model_LR_r2 = r2_score(y_test, y_pred)

print(model_LR_MAE)
print(model_LR_RMSE)
print(model_LR_r2)

0.5130730108879092
0.7220715230668358
0.6057158459716574


In [64]:
#RandomForestRegressor baseline model

model_RFR = Pipeline([
    ('preprocessor', preprocessor),
    ('randomforest', RandomForestRegressor(n_estimators=100,
                                           random_state=42
    ))
])

model_RFR.fit(X_train, y_train)
y_pred = model_RFR.predict(X_test)

model_RFR_MAE = mean_absolute_error(y_test, y_pred)
model_RFR_MSE = mean_squared_error(y_test, y_pred)
model_RFR_RMSE = np.sqrt(model_RFR_MSE)
model_RFR_r2 = r2_score(y_test, y_pred)

print(model_RFR_MAE)
print(model_RFR_RMSE)
print(model_RFR_r2)

print("Training R2:", model_RFR.score(X_train, y_train))
print("Test R2:", model_RFR.score(X_test, y_test))

0.3324132445494188
0.5209504572742295
0.7947695793855807
Training R2: 0.9737973193938616
Test R2: 0.7947695793855807


In [65]:
#DecisionTreeRegressor baseline model

model_DTR = Pipeline([
    ('preprocessor', preprocessor),
    ('decisiontree', DecisionTreeRegressor(random_state=42))
])

model_DTR.fit(X_train, y_train)
y_pred = model_DTR.predict(X_test)

model_DTR_MAE = mean_absolute_error(y_test, y_pred)
model_DTR_MSE = mean_squared_error(y_test, y_pred)
model_DTR_RMSE = np.sqrt(model_DTR_MSE)
model_DTR_r2 = r2_score(y_test, y_pred)

print(model_DTR_MAE)
print(model_DTR_RMSE)
print(model_DTR_r2)

print("Training R2: ", model_DTR.score(X_train, y_train))
print("Test R2: ", model_DTR.score(X_test, y_test))

0.453369878875969
0.716012695590486
0.6123048806722347
Training R2:  1.0
Test R2:  0.6123048806722347


In [71]:
#MLPRegressor baseline model

model_MLP = Pipeline([
    ('preprocessing', preprocessor),
    ('MLP', MLPRegressor(random_state=42))
])

model_MLP.fit(X_train, y_train)
y_pred = model_MLP.predict(X_test)

model_MLP_MAE = mean_absolute_error(y_test, y_pred)
model_MLP_MSE = mean_squared_error(y_test, y_pred)
model_MLP_RMSE = np.sqrt(model_MLP_MSE)
model_MLP_r2 = r2_score(y_test, y_pred)

print(model_MLP_MAE)
print(model_MLP_RMSE)
print(model_MLP_r2)

print("Training R2: ", model_MLP.score(X_train, y_train))
print("Test R2: ", model_MLP.score(X_test, y_test))

0.3643254893407912
0.543640341457103
0.7765027204320781
Training R2:  0.8091784901927993
Test R2:  0.7765027204320781


C:\Users\Russell\anaconda3\envs\sp500\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [74]:
#GradientBoostingRegressor basleine model

model_GBR = Pipeline([
    ("preprocessor", preprocessor),
    ("gradientBR", GradientBoostingRegressor(random_state=42))
])

model_GBR.fit(X_train, y_train)
y_pred = model_GBR.predict(X_test)

model_GBR_MAE = mean_absolute_error(y_test, y_pred)
model_GBR_MSE = mean_squared_error(y_test, y_pred)
model_GBR_RMSE = np.sqrt(model_GBR_MSE)
model_GBR_r2 = r2_score(y_test, y_pred)

print(model_GBR_MAE)
print(model_GBR_RMSE)
print(model_GBR_r2)

print("Training R2: ", model_GBR.score(X_train, y_train))
print("Test R2: ", model_GBR.score(X_test, y_test))

0.37531599977127644
0.5472369490012192
0.7735357184269718
Training R2:  0.8081429241877414
Test R2:  0.7735357184269718


## Baseline Models Analysis

LinearRegression provides a baseline, with R2 of 0.606

DecisionTree only slightly better than LinearRegression, with a R2 of 0.612, and significantly overfitting with a R2 of 1 on training

RandomForest is the bestperforming baseline, R2 of 0.795, but with a big gap between training R2 of 0.974 and test R2, indicting overfitting

MLPRegressor and Gradient Boosting performed similary, with tes R2 of 0.777 and 0.774, with relatively smaller training-test gaps